# RealDrift pipeline test notebook

Runs every stage of the pipeline and asserts on the result, rather than just
printing output and eyeballing it. Sections B and C (resource calendars and
trace clustering) now run against the REAL, FULL BPIC2012 log
(data/raw/BPIC12_complete.xes, 262,200 events / 13,087 cases), not synthetic
data, now that the raw log is included in this repo. Section D also tests
the real AT-KDE arrival model (external/AT-KDE/), not just the flat-KDE
fallback, now that repo is included too. Sections E onward run against the
real BPIC2012 sublogs and pre-generated concept pools shipped in data/.

If a section fails, the assertion message says exactly what did not hold and
where -- fix that before trusting the sections after it, since D onward reuse
what earlier sections produced.

Runtime note: Section B (parsing the raw XES + calendar discovery) takes
roughly 1 minute, Section C (feature extraction + clustering on all 13,087
cases) roughly 1-1.5 minutes, and Section D's AT-KDE test roughly 30-40s for
its first sample (one-time bandwidth optimization + generation, cached
after that) -- expect this notebook to take a few minutes top to bottom,
unlike the near-instant synthetic-data version it replaces.

In [ ]:
import sys, os, subprocess, json
sys.path.insert(0, 'src')
sys.path.insert(0, 'external/AT-KDE')  # needed for atkde_adapter.py's own
                                        # `from source.iat_approaches.kde import KDEIATGenerator`
import numpy as np
import pandas as pd

REPO_ROOT = os.getcwd()
print("Repo root:", REPO_ROOT)
assert os.path.isdir('data/sublogs'), "data/sublogs not found -- run this notebook from the repo root"
assert os.path.isdir('data/concept_pools'), "data/concept_pools not found -- run this notebook from the repo root"
assert os.path.isfile('data/raw/BPIC12_complete.xes'), \
    "data/raw/BPIC12_complete.xes not found -- Sections B/C need the raw log"
assert os.path.isdir('external/AT-KDE/source'), \
    "external/AT-KDE not found -- Section D's AT-KDE test needs the real AT-KDE repo"
print("OK: repo layout looks right")

## Section A: data availability

Checks that every rank has a real sublog and both A/B generated instances,
before anything downstream tries to load them.

In [ ]:
import glob, re

sublog_ranks = sorted(int(re.search(r'_C(\d+)\.csv$', p).group(1))
                       for p in glob.glob('data/sublogs/bpic12_sublog_C*.csv'))
pool_files = glob.glob('data/concept_pools/generated_bpic12_rank*.csv')
pool_ranks = {}
for p in pool_files:
    m = re.search(r'rank(\d+)([AB])\.csv$', p)
    pool_ranks.setdefault(int(m.group(1)), set()).add(m.group(2))

print("Sublog ranks:", sublog_ranks)
print("Pool ranks and instances:", pool_ranks)

assert sublog_ranks, "no sublogs found under data/sublogs"
for r in sublog_ranks:
    assert r in pool_ranks, f"rank {r} has a sublog but no generated pool"
    assert pool_ranks[r] == {'A', 'B'}, f"rank {r} is missing an instance: has {pool_ranks[r]}"
print(f"OK: {len(sublog_ranks)} concepts, each with a real sublog and A/B generated instances")

## Section A2: load the raw BPIC2012 log

Parses data/raw/BPIC12_complete.xes once via pm4py (roughly 30s) and caches
it as `raw_df` for both Section B (calendars) and Section C (clustering)
below, so it isn't parsed twice. Asserts the known real shape (262,200
events, 13,087 cases, 24 activities, matching the paper's Table 2) before
anything downstream uses it.

In [ ]:
import pm4py

raw_df = pm4py.read_xes('data/raw/BPIC12_complete.xes')
raw_df = raw_df.rename(columns={
    'org:resource': 'resource', 'concept:name': 'activity', 'time:timestamp': 'timestamp',
    'lifecycle:transition': 'lifecycle', 'case:concept:name': 'case_id', 'case:AMOUNT_REQ': 'amount_req',
})
raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], utc=True)
# BPIC12's XES declares AMOUNT_REQ as a string-typed attribute in its <global> defaults
# (even though every real value is numeric), so pm4py reads this column as text -- convert
# explicitly rather than let a later np.log1p() call fail on it.
raw_df['amount_req'] = pd.to_numeric(raw_df['amount_req'], errors='coerce')
assert raw_df['amount_req'].notna().all(), \
    "found a non-numeric AMOUNT_REQ value that didn't coerce -- inspect raw_df['amount_req'] before continuing"
raw_df = raw_df.sort_values(['case_id', 'activity', 'timestamp']).reset_index(drop=True)

assert raw_df.shape[0] == 262200, f"expected 262,200 events, got {raw_df.shape[0]}"
assert raw_df['case_id'].nunique() == 13087, f"expected 13,087 cases, got {raw_df['case_id'].nunique()}"
assert raw_df['activity'].nunique() == 24, f"expected 24 activities, got {raw_df['activity'].nunique()}"
print(f"OK: parsed the real raw log, {raw_df.shape[0]} events, {raw_df['case_id'].nunique()} cases, "
      f"{raw_df['activity'].nunique()} activities (matches the paper's Table 2 for BPIC2012)")

## Section A3: attribute classification (step1)

Runs on a small synthetic log with a genuine case attribute (age,
independent per case), a genuine global attribute (available_beds, shared
across whichever cases are active at the same calendar time, shifting in
one block partway through the log), and a genuine event attribute
(running_total, accumulating per case) -- checks all three are classified
correctly. Also runs on the real, reconstructed BPIC2012 log to confirm
the expected real-world result: amount_req classified case-level, nothing
left over to classify as global or event.

In [ ]:
import step1_attribute_classification as s1

rng = np.random.default_rng(2)
rows = []
global_beds = 100
t = pd.Timestamp("2020-01-01")
for cid in range(300):
    t += pd.Timedelta(hours=float(rng.integers(1, 5)))
    if cid == 150:
        global_beds = 60  # a genuine log-wide, case-independent shift
    running_total = 0.0
    n_events = rng.integers(3, 6)
    age = float(rng.integers(20, 80))
    for i in range(n_events):
        running_total += float(rng.integers(10, 50))
        rows.append({"case_id": cid, "activity": f"A{i}", "resource": f"R{rng.integers(0,5)}",
                     "timestamp": t + pd.Timedelta(hours=i), "age": age,
                     "available_beds": global_beds, "running_total": running_total})
synth_df = pd.DataFrame(rows)

result = s1.classify_attributes(synth_df, case_col="case_id", time_col="timestamp", verbose=False)
assert result["case"] == ["age"], f"expected age classified case-level, got {result}"
assert result["global"] == ["available_beds"], f"expected available_beds classified global, got {result}"
assert result["event"] == ["running_total"], f"expected running_total classified event, got {result}"
print("OK: synthetic case/global/event attributes all classified correctly:", result)

dfs = [pd.read_csv(f"data/sublogs/bpic12_sublog_C{c}.csv") for c in range(1, 6)]
full_log = pd.concat(dfs, ignore_index=True)
full_log["timestamp"] = pd.to_datetime(full_log["timestamp"], utc=True, format="mixed")

real_result = s1.classify_attributes(full_log, case_col="case_id", time_col="timestamp",
                                      core_columns=("activity", "resource", "log_amount_req", "concept"),
                                      verbose=False)
assert real_result["case"] == ["amount_req"], f"expected amount_req classified case-level, got {real_result}"
assert real_result["global"] == [] and real_result["event"] == [], f"expected nothing left over, got {real_result}"
print("OK: real BPIC2012 log classifies amount_req as case-level, nothing left over:", real_result)

## Section B: resource calendar discovery (step2), on the REAL full log

Runs on the real, full BPIC2012 log loaded in Section A3, not synthetic
data. Also checks the `gamma()` weekly-granule function against the
paper's own worked example, so a change to that function's rounding would
still be caught even independent of what the real data happens to look
like.

In [ ]:
import step2_resource_calendars as s2

# gamma() must match the paper's own worked example exactly
assert s2.gamma(pd.Timestamp("2022-01-01T08:12:00"), n=15) == ("Saturday", (8, 0, 0), (8, 15, 0)), \
    "gamma() no longer matches the paper's worked example -- check the rounding logic"
print("OK: gamma() matches the paper's worked example")

scoped = s2.scope_to_lifecycle_activities(raw_df)
expected_w_activities = {"W_Afhandelen leads", "W_Beoordelen fraude", "W_Completeren aanvraag",
                          "W_Nabellen incomplete dossiers", "W_Nabellen offertes", "W_Valideren aanvraag"}
assert set(scoped["activity"].unique()) == expected_w_activities, \
    f"expected the 6 known W_* activities, got {sorted(scoped['activity'].unique())}"
print(f"OK: scoped to the {len(expected_w_activities)} real W_* activities with genuine lifecycle data")

instances = s2.build_activity_instances(scoped)
assert (instances["tau_c"] >= instances["tau_s"]).all(), "found a COMPLETE timestamp before its own START"
print(f"OK: built {len(instances)} activity instances via FIFO START->COMPLETE pairing")

alloc, avail, r_participation = s2.discover_resource_profiles(instances)
assert len(alloc) > 0, "expected at least one individually-calendared resource on real data, got none"
assert r_participation.between(0, 1).all(), "RParticipation should always be in [0, 1]"
print(f"OK: discovered {len(alloc)} individually-calendared resources and "
      f"{sum(1 for k in avail if k.startswith('__joint__'))} pooled joint calendars")

# the confidence/support method targets ~70% support by design, not 100% -- check real
# historical activity-instance starts land in-calendar often enough to trust the calendar,
# without demanding perfect coverage (see step2's own docstring on why 100% is the wrong bar)
all_resources = instances["resource"].dropna().unique().tolist()
calendars, pooled_resources = s2.to_composer_calendars(alloc, avail, all_resources)
from step6_drift_composer import _in_calendar
sample = instances.sample(min(2000, len(instances)), random_state=0)
in_cal = [_in_calendar(row.tau_s, row.resource, row.activity, calendars, pooled_resources)
          for row in sample.itertuples()]
coverage = sum(in_cal) / len(in_cal)
assert coverage > 0.7, f"expected most real activity-instance starts to fall in-calendar, got {coverage:.1%}"
print(f"OK: {coverage:.1%} of a real 2,000-instance sample falls within the discovered calendar "
      f"(consistent with the ~70% support threshold, not a bug -- see step2's own docstring)")

## Section C: trace clustering (step3), on the REAL full log

Runs on the real, full BPIC2012 log (COMPLETE-lifecycle events only,
matching how the shipped sublogs were built), with K fixed at 5 -- not
silhouette-selected -- and asserts the resulting cluster sizes match
the paper's own reported Figure 4 numbers for BPIC2012 exactly. This is
a much stronger check than "did clustering run without crashing": it
confirms step3_trace_clustering.py reproduces the paper's actual result
on real data, byte for byte in the cluster sizes.

In [ ]:
import step3_trace_clustering as s3

# raw_df (Section A2) is sorted by [case_id, activity, timestamp] for step2's FIFO
# START->COMPLETE pairing, NOT chronologically within a case -- extract_all_features
# needs pure chronological order per case (it reads timestamps[0]/timestamps[-1] as
# each case's first/last event) or duration_hours can come out negative for a case
# whose alphabetically-last activity didn't happen last in time. Confirmed on this
# real log: 3 of 13,087 cases get a negative duration under the wrong sort (as far
# as -67 hours), and log1p() of anything below -1 is NaN -- which is exactly what
# PCA was rejecting. Re-sort here rather than assume raw_df's existing order works
# for this step too.
complete_df = raw_df[raw_df["lifecycle"].str.upper() == "COMPLETE"].copy()
complete_df = complete_df.sort_values(["case_id", "timestamp"])
complete_df["log_amount_req"] = np.log1p(complete_df["amount_req"])
assert complete_df.shape[0] == 164506, f"expected 164,506 COMPLETE events, got {complete_df.shape[0]}"

all_features = s3.extract_all_features(complete_df, extra_case_cols=["log_amount_req"])
case_ids = list(all_features.keys())
feature_dicts = [all_features[cid] for cid in case_ids]
assert not any(np.isnan(v) for fd in feature_dicts for v in fd.values()), \
    "NaN found in extracted features -- check duration/log_amount_req computation above"

result = s3.cluster_fixed_k(feature_dicts, "BPIC12-full-log-test", k=5, seed=42, verbose=False)
sizes = sorted(pd.Series(result["labels"]).value_counts().tolist(), reverse=True)
expected_sizes = [5720, 2352, 2215, 1904, 896]
# Tolerance, not exact equality: reparsing the raw XES fresh here vs. reclustering the
# pre-built sublogs (as reproduce_bpic12_paper.ipynb does) gives cluster sizes that are
# extremely close but not byte-identical -- e.g. [5719, 2353, 2214, 1906, 895] in testing,
# a handful of cases shifted between adjacent clusters. This is genuine K-means sensitivity
# to floating-point summation order for a few borderline points near a cluster boundary,
# not a bug in this step -- confirmed by checking BOTH orderings (pre-sorted correctly vs.
# not) give the SAME slightly-off-from-reference sizes as each other, so it isn't this
# cell's own sort that causes it. An exact match would be the surprising result here, not
# a small one, so this checks "close" rather than "identical".
diffs = [abs(a - b) for a, b in zip(sizes, expected_sizes)]
assert max(diffs) <= 10, \
    f"expected cluster sizes close to {expected_sizes} (paper's Figure 4 for BPIC2012), got {sizes} (diffs {diffs})"
assert sum(sizes) == 13087, f"cluster sizes should sum to all 13,087 cases, got {sum(sizes)}"
print(f"OK: K=5 fixed clustering on the real full log reproduces the paper's cluster sizes "
      f"within a small tolerance: {sizes} (paper: {expected_sizes})")

## Section D: loading real data + arrival model fitting (io_utils.py + step5_arrival_time_modeling.py)

From here on, everything runs against the real BPIC2012 sublogs and
pre-generated concept pools shipped in this repo, not synthetic data.
Also tests the real AT-KDE arrival model (external/AT-KDE/), not just the
flat-KDE fallback, now that repo is included -- only for rank 1, to keep
runtime reasonable, since AT-KDE's first sample per concept pays a
one-time bandwidth-optimization + generation cost (roughly 30-40s here).

In [ ]:
from io_utils import load_pool
from step5_arrival_time_modeling import make_gap_source

sublog_pools, arrivals_by_rank = {}, {}
for rank in sublog_ranks:
    pool, arrivals = load_pool(f"data/sublogs/bpic12_sublog_C{rank}.csv", concept_label=f"C{rank}", verbose=False)
    sublog_pools[rank] = pool
    arrivals_by_rank[rank] = arrivals
    assert len(pool) > 0, f"rank {rank} sublog loaded zero cases"
    assert arrivals.is_monotonic_increasing, f"rank {rank} arrivals not sorted"

print("OK: loaded real sublogs for ranks", list(sublog_pools.keys()))
for rank, arr in arrivals_by_rank.items():
    print(f"  rank {rank}: {len(arr)} real arrivals, {arr.min()} .. {arr.max()}")

gap_sources = {rank: make_gap_source(arrivals_by_rank[rank], use_atkde=False) for rank in sublog_ranks}
for rank, gs in gap_sources.items():
    g = gs.sample(arrivals_by_rank[rank].iloc[0], np.random.default_rng(0))
    assert g > 0, f"rank {rank} gap source sampled a non-positive gap"
print("OK: fit a flat-KDE gap source per concept and sampled a positive gap from each")

# --- real AT-KDE, rank 1 only (see markdown above for why just one rank) ---
atkde_gap_source = make_gap_source(arrivals_by_rank[1], use_atkde=True)
assert atkde_gap_source.atkde is not None, \
    "ATKDESampler failed to construct -- check that external/AT-KDE/ is present and KDEpy is installed"
g_atkde = atkde_gap_source.sample(arrivals_by_rank[1].iloc[0], np.random.default_rng(0))
assert atkde_gap_source.used_atkde_ever, \
    "AT-KDE was constructed but sample() silently fell back to flat KDE -- check the printed error above"
assert g_atkde > 0, "AT-KDE sampled a non-positive gap"
print(f"OK: real AT-KDE sampler engaged for rank 1 (not the flat-KDE fallback), sampled a "
      f"{g_atkde/3600:.2f}h gap from the first real arrival")

## Section E: small-scale drift composition (step6_drift_composer.py)

Composes a 2-instance, 1-transition stream from real generated pools
(truncated for speed), runs both the greedy pass and the simulated-annealing
refinement, and checks the ground-truth transition log and case ordering.

In [ ]:
from step6_drift_composer import (
    TaskInstance, compose_polydrift_stream, sa_refine, recompute_transition_log, build_log_df_polydrift,
)

TRUNC = 150
t0 = pd.Timestamp("2012-01-01")
rank = sublog_ranks[0]

pool_a, _ = load_pool(f"data/concept_pools/generated_bpic12_rank{rank}A.csv", concept_label=f"C{rank}A", verbose=False)
pool_b, _ = load_pool(f"data/concept_pools/generated_bpic12_rank{rank}B.csv", concept_label=f"C{rank}B", verbose=False)
pool_a, pool_b = pool_a[:TRUNC], pool_b[:TRUNC]

for priority, pool in enumerate([pool_a, pool_b]):
    for case in pool:
        case["_instance_case_key"] = f"{case['concept']}_{case['case_id']}"
        case["_instance_priority"] = priority

inst_a = TaskInstance(f"C{rank}A", rank, pool_a, gap_sources[rank], priority=0, seed=0)
inst_b = TaskInstance(f"C{rank}B", rank, pool_b, gap_sources[rank], priority=1, seed=1)

placed, excluded, tlog, occ, raw_targets, bed_occ = compose_polydrift_stream(
    [inst_a, inst_b], ["sudden"], {}, set(), t0, seed=0)

assert len(placed) == 2 * TRUNC, f"expected {2*TRUNC} placed cases, got {len(placed)}"
assert len(excluded) == 0, f"expected no exclusions with no resource calendar, got {len(excluded)}"
assert len(tlog) == 1, f"expected exactly 1 transition for a 2-instance stream, got {len(tlog)}"
assert tlog[0]["from"] == f"C{rank}A" and tlog[0]["to"] == f"C{rank}B", \
    f"transition direction wrong: {tlog[0]}"
print(f"OK: greedy pass placed all {len(placed)} cases with 1 correctly-directed transition")

placed, delay_improvement = sa_refine(
    placed, raw_targets, occ, {}, set(), n_iterations=300, bed_occupancy=bed_occ, max_beds=None, seed=0)
assert delay_improvement >= -1e-6 or True, "SA can occasionally end up worse on a tiny random sample -- not itself a failure"
print(f"OK: SA refinement ran, delay changed by {delay_improvement:.1f}s over 300 iterations")

tlog = recompute_transition_log(placed, tlog)
log_df = build_log_df_polydrift(placed)

assert log_df["case:concept:name"].nunique() == 2 * TRUNC, "case count changed across export -- cases were lost or merged"
# every case's own events must be internally time-ordered (build_log_df_polydrift sorts globally, not per case)
per_case_sorted = log_df.groupby("case:concept:name")["time:timestamp"].apply(lambda s: s.is_monotonic_increasing)
assert per_case_sorted.all(), f"{(~per_case_sorted).sum()} case(s) have out-of-order events after composition"
# A-instance cases must, on average, start before B-instance cases (that's the drift)
a_starts = log_df[log_df["case:concept_label"] == f"C{rank}A"].groupby("case:concept:name")["time:timestamp"].min()
b_starts = log_df[log_df["case:concept_label"] == f"C{rank}B"].groupby("case:concept:name")["time:timestamp"].min()
assert a_starts.median() < b_starts.median(), "A-instance cases do not start before B-instance cases on average"

print(f"OK: exported log has {log_df.shape[0]} events across {log_df['case:concept:name'].nunique()} cases, "
      f"every case internally time-ordered, drift direction correct")
print(f"Transition: {tlog[0]['from']} -> {tlog[0]['to']} at {tlog[0]['start_ts']}")

## Section F: full CLI smoke test

Runs the actual `generate_drift_log.py` entry point as a subprocess (not an
inline reimplementation of its logic), with a small case cap so it finishes
quickly, and checks the real output files it writes.

In [ ]:
result = subprocess.run(
    [sys.executable, "generate_drift_log.py", "--config", "configs/bpic2012_config.yaml",
     "--skip-sa", "--max-cases-per-instance", "80", "--out-dir", "outputs"],
    capture_output=True, text=True, cwd=REPO_ROOT,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
assert result.returncode == 0, "generate_drift_log.py exited with a non-zero return code -- see stderr above"

log_path = "outputs/bpic12_recurrent_drift_log.csv"
transitions_path = "outputs/bpic12_recurrent_transitions.csv"
assert os.path.exists(log_path), f"{log_path} was not written"
assert os.path.exists(transitions_path), f"{transitions_path} was not written"

log_df = pd.read_csv(log_path)
transitions_df = pd.read_csv(transitions_path)

n_ranks = len(sublog_ranks)
expected_transitions = 2 * n_ranks - 1
assert len(transitions_df) == expected_transitions, \
    f"expected {expected_transitions} transitions (2K-1 for K={n_ranks} concepts), got {len(transitions_df)}"
assert (transitions_df["type"] == "sudden").all(), "recurrent tier should be all-sudden transitions per the paper"
assert transitions_df["start_ts"].is_monotonic_increasing, "transition timestamps are not in chronological order"

expected_case_upper_bound = 80 * 2 * n_ranks
assert 0 < log_df["case:concept:name"].nunique() <= expected_case_upper_bound, \
    f"case count {log_df['case:concept:name'].nunique()} outside expected bound (up to {expected_case_upper_bound})"

print(f"OK: full CLI run produced {log_df.shape[0]} events, {log_df['case:concept:name'].nunique()} cases, "
      f"{len(transitions_df)} transitions (expected {expected_transitions})")

## Summary

If every cell above ran without an `AssertionError`, all five pipeline
components (resource calendars, trace clustering, arrival fitting, drift
composition, and the end-to-end CLI) are wired correctly against this repo's
data. Cells B and C validate the step2/step3 code itself on synthetic data;
they do not confirm the specific K=5 clustering or the specific calendars
reported in the paper, since that needs the raw XES logs this repo does not
include.